# API Behavioral Threat Detection - Staged Experiments (v2)

This notebook explains each stage clearly and saves report-ready tables and figures.

## 0. Setup

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler, StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB

from imblearn.over_sampling import RandomOverSampler, SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.combine import SMOTEENN

try:
    from xgboost import XGBClassifier
except Exception:
    XGBClassifier = None

BASE = Path('..').resolve()
DATA_RAW = BASE / 'data' / 'raw' / 'behavioral_api_dataset_v2.csv'
DATA_PROC = BASE / 'data' / 'processed'
DATA_IMB = BASE / 'data' / 'imbalanced'
REPORTS = BASE / 'reports'
FIGS = REPORTS / 'figures'

for p in [DATA_PROC, DATA_IMB, REPORTS, FIGS]:
    p.mkdir(parents=True, exist_ok=True)

FEATURES = [
    'inter_api_access_duration(sec)', 'api_access_uniqueness', 'sequence_length(count)',
    'vsession_duration(min)', 'ip_type', 'num_sessions', 'num_users',
    'num_unique_apis', 'source', 'failed_auth_count', 'token_reuse_ratio',
    'status_4xx_ratio', 'status_5xx_ratio'
]
TARGET = 'attack_class'
CAT_COLS = ['ip_type', 'source']
NUM_COLS = [c for c in FEATURES if c not in CAT_COLS]

def make_preprocessor(scale='none'):
    if scale == 'minmax':
        num_pipe = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', MinMaxScaler())])
    elif scale == 'standard':
        num_pipe = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())])
    else:
        num_pipe = Pipeline([('imputer', SimpleImputer(strategy='median'))])
    cat_pipe = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('encoder', OneHotEncoder(handle_unknown='ignore'))])
    return ColumnTransformer([('num', num_pipe, NUM_COLS), ('cat', cat_pipe, CAT_COLS)])

def metrics_row(stage, resampling, scaling, model_name, y_true, y_pred):
    return {
        'stage': stage, 'resampling': resampling, 'scaling': scaling, 'model': model_name,
        'accuracy': round(accuracy_score(y_true, y_pred), 4),
        'macro_precision': round(precision_score(y_true, y_pred, average='macro', zero_division=0), 4),
        'macro_recall': round(recall_score(y_true, y_pred, average='macro', zero_division=0), 4),
        'macro_f1': round(f1_score(y_true, y_pred, average='macro', zero_division=0), 4),
    }

def apply_resampling(name, X, y):
    if name == 'none':
        return X.copy(), y.copy()
    samplers = {
        'oversampling': RandomOverSampler(random_state=42),
        'undersampling': RandomUnderSampler(random_state=42),
        'smote': SMOTE(random_state=42),
        'smoteenn': SMOTEENN(random_state=42),
    }
    return samplers[name].fit_resample(X, y)


## 1. Load dataset and split train/test

In [ ]:
df = pd.read_csv(DATA_RAW)
X = df[FEATURES].copy()
y = df[TARGET].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train.to_csv(DATA_PROC / 'X_train.csv', index=False)
X_test.to_csv(DATA_PROC / 'X_test.csv', index=False)
y_train.to_frame(name=TARGET).to_csv(DATA_PROC / 'y_train.csv', index=False)
y_test.to_frame(name=TARGET).to_csv(DATA_PROC / 'y_test.csv', index=False)

print('Train:', X_train.shape, 'Test:', X_test.shape)
print(y_train.value_counts())

# class distribution chart
vc = y_train.value_counts().sort_index()
plt.figure(figsize=(7,4))
vc.plot(kind='bar')
plt.title('Train Class Distribution')
plt.ylabel('Count')
plt.tight_layout()
plt.savefig(FIGS / 'class_distribution_train.png', dpi=160)
plt.show()


## 2. Stage A - Resampling filter (RandomForest baseline)

In [ ]:
stage_a_rows = []
resampling_methods = ['none', 'oversampling', 'undersampling', 'smote', 'smoteenn']

for r in resampling_methods:
    Xr, yr = apply_resampling(r, X_train, y_train)
    pd.DataFrame(Xr, columns=FEATURES).to_csv(DATA_IMB / f'X_{r}.csv', index=False)
    pd.DataFrame({TARGET: yr}).to_csv(DATA_IMB / f'y_{r}.csv', index=False)

    pipe = Pipeline([
        ('preprocessor', make_preprocessor('none')),
        ('model', RandomForestClassifier(n_estimators=220, random_state=42, class_weight='balanced'))
    ])
    pipe.fit(Xr, yr)
    yp = pipe.predict(X_test)
    stage_a_rows.append(metrics_row('A_resampling_filter', r, 'none', 'RandomForest', y_test, yp))

stage_a_df = pd.DataFrame(stage_a_rows).sort_values(['macro_f1', 'macro_recall'], ascending=False)
stage_a_df.to_csv(REPORTS / 'stage_a_resampling_results.csv', index=False)
print(stage_a_df)

plt.figure(figsize=(8,4))
plt.bar(stage_a_df['resampling'], stage_a_df['macro_f1'])
plt.title('Stage A: Macro F1 by Resampling')
plt.ylabel('Macro F1')
plt.tight_layout()
plt.savefig(FIGS / 'stage_a_macro_f1.png', dpi=160)
plt.show()

top2 = stage_a_df.head(2)['resampling'].tolist()
print('Top-2 resampling:', top2)


## 3. Stage B - Scaling filter on top-2 resampling

In [ ]:
stage_b_rows = []
scalers = ['none', 'minmax', 'standard']

for r in top2:
    Xr, yr = apply_resampling(r, X_train, y_train)
    for s in scalers:
        pipe = Pipeline([
            ('preprocessor', make_preprocessor(s)),
            ('model', RandomForestClassifier(n_estimators=220, random_state=42, class_weight='balanced'))
        ])
        pipe.fit(Xr, yr)
        yp = pipe.predict(X_test)
        stage_b_rows.append(metrics_row('B_scaling_filter', r, s, 'RandomForest', y_test, yp))

stage_b_df = pd.DataFrame(stage_b_rows).sort_values(['macro_f1', 'macro_recall'], ascending=False)
stage_b_df.to_csv(REPORTS / 'stage_b_scaling_results.csv', index=False)
print(stage_b_df)

best = stage_b_df.iloc[0]
best_resampling, best_scaling = best['resampling'], best['scaling']
print('Best combo:', best_resampling, best_scaling)

pivot_b = stage_b_df.pivot(index='resampling', columns='scaling', values='macro_f1')
plt.figure(figsize=(7,4))
for idx in pivot_b.index:
    plt.plot(pivot_b.columns, pivot_b.loc[idx], marker='o', label=idx)
plt.title('Stage B: Macro F1 by Scaling (Top Resampling)')
plt.ylabel('Macro F1')
plt.legend()
plt.tight_layout()
plt.savefig(FIGS / 'stage_b_scaling_lines.png', dpi=160)
plt.show()


## 4. Stage C - Final model comparison

In [ ]:
Xr, yr = apply_resampling(best_resampling, X_train, y_train)

models = {
    'RandomForest': RandomForestClassifier(n_estimators=300, random_state=42, class_weight='balanced'),
    'SVM': SVC(probability=True, class_weight='balanced', random_state=42),
    'LogisticRegression': LogisticRegression(max_iter=1500, class_weight='balanced'),
    'NaiveBayes': GaussianNB(),
}
if XGBClassifier is not None:
    models['XGBoost'] = XGBClassifier(
        eval_metric='mlogloss', random_state=42, n_estimators=250,
        max_depth=6, learning_rate=0.08, subsample=0.9, colsample_bytree=0.9
    )

stage_c_rows = []
trained = {}
for mname, model in models.items():
    pipe = Pipeline([('preprocessor', make_preprocessor(best_scaling)), ('model', model)])
    pipe.fit(Xr, yr)
    yp = pipe.predict(X_test)
    stage_c_rows.append(metrics_row('C_model_comparison', best_resampling, best_scaling, mname, y_test, yp))
    trained[mname] = pipe

stage_c_df = pd.DataFrame(stage_c_rows).sort_values(['macro_f1', 'macro_recall'], ascending=False)
stage_c_df.to_csv(REPORTS / 'stage_c_model_comparison.csv', index=False)
stage_c_df.head(1).to_csv(REPORTS / 'model_results_v2.csv', index=False)
print(stage_c_df)

plt.figure(figsize=(8,4))
plt.bar(stage_c_df['model'], stage_c_df['macro_f1'])
plt.title('Stage C: Macro F1 by Model')
plt.ylabel('Macro F1')
plt.tight_layout()
plt.savefig(FIGS / 'stage_c_model_macro_f1.png', dpi=160)
plt.show()

best_model_name = stage_c_df.iloc[0]['model']
best_pipeline = trained[best_model_name]
print('Best model:', best_model_name)


## 5. Final confusion matrix + save model

In [ ]:
from sklearn.metrics import classification_report

y_pred = best_pipeline.predict(X_test)
report_text = classification_report(y_test, y_pred, digits=4)
cm = confusion_matrix(y_test, y_pred, labels=sorted(y_test.unique()))

(REPORTS / 'classification_report_v2.txt').write_text(report_text, encoding='utf-8')

fig, ax = plt.subplots(figsize=(6,5))
ConfusionMatrixDisplay(cm, display_labels=sorted(y_test.unique())).plot(ax=ax, xticks_rotation=20, colorbar=False)
plt.title('Best Model Confusion Matrix')
plt.tight_layout()
plt.savefig(FIGS / 'stage_c_confusion_matrix.png', dpi=160)
plt.show()

summary = pd.DataFrame([{
    'top2_resampling': ', '.join(top2),
    'best_resampling': best_resampling,
    'best_scaling': best_scaling,
    'best_model': best_model_name
}])
summary.to_csv(REPORTS / 'final_selection_summary.csv', index=False)

joblib.dump(best_pipeline, BASE / 'model' / 'api_security_pipeline_v2.pkl')
print(report_text)
print('Saved model + reports + figures.')


## 6. Report-ready outputs checklist

- `reports/stage_a_resampling_results.csv`
- `reports/stage_b_scaling_results.csv`
- `reports/stage_c_model_comparison.csv`
- `reports/final_selection_summary.csv`
- `reports/classification_report_v2.txt`
- `reports/figures/*.png`

Use these directly in your black-book results chapter.